## Data Loading and cleaning
loading the job salary dataset and performing initial checks to confirm data quality before analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from google.colab import drive
drive.mount('/content/drive')
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pickle


In [ ]:
!find / content/drive -iname 'job_salary_prediction_dataset.csv'

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Ml Project /job_salary_prediction_dataset.csv')
df.shape
df.info()
df.isnull().sum()
print ("Number of duplicates: ", df.duplicated().sum())

## Question one: Which industry pay the most, and which pay the least?

In [ ]:
industry_salary = df.groupby('industry')['salary'].agg(['mean', 'median']).sort_values('mean', ascending=False)
industry_salary

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=industry_salary.index, y=industry_salary['mean'])
plt.xlabel('Industry')
plt.ylabel('Mean Salary')
plt.title('Average Salary by Industry')
plt.xticks(rotation=4, ha='right')
plt.show()

###Statistical Test

In [ ]:
from scipy.stats import f_oneway

groups = [df[df['industry'] == i]['salary'] for i in df['industry'].unique()]
f_stat, p_value = f_oneway(*groups)
print ("f statistic: ", f_stat)
print ("p value: ", p_value)

Mean salaries across industries range normally with no industry paying
dramatically more or less than others (visually confirmed in the bar chart)
ANOVA results: F statistic = 0.75 and P Value = 0.66
Since P Value is far above 0.05 threshold, this difference is not statistically significant.
Industry does not meaningfully predict salary in this dataset

#Question 2
Does more education (Bsc > MSc > Phd) mean more salary?

In [ ]:
education_salary = df.groupby('education_level')['salary'].mean().sort_values(ascending=False) ##sorting te mean of each education level
education_salary

In [ ]:
plt.figure(figsize=(8,6))
sns.barplot(data=df, x='education_level', y='salary')
plt.xlabel('Education Level')
plt.ylabel('Salary')
plt.title('Salary by Education Level')
plt.show()

In [ ]:
#statistical test to confirm whether salary differs significantly across education levels
groups_edu = [df[df['education_level'] == e]['salary'] for e in df['education_level'].unique()]
f_stat_edu, p_value_edu = f_oneway(*groups_edu)
print("f statistic: ", f_stat_edu)
print("p value: ", p_value_edu)

###
* Mean salary increases significantly with education level, with Phd as the highest. Phd earns 24% more than High School


*   ANOVA: f Statistic = 6624.7914, p value = 0
*   Since p value is far below the 0.05 threshold, this difference is highly statistically significant. Therefore education level meaningfully predicts salary in this dataset
*   Aditional level of education equals additional increase in salary

# Question 3
Does location matter? Do jobs in big cities pay more than remote or rural jobs?

In [ ]:
#grouping salary by location
location_salary = df.groupby('location')['salary'].mean().sort_values(ascending=False)
location_salary

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=location_salary.index, y=location_salary)
plt.xlabel('Location')
plt.ylabel('Mean Salary')
plt.title('Average Salary by Location')
plt.show()

In [ ]:
#statistical test for location
groups_loc = [df[df['location'] == l]['salary'] for l in df['location'].unique()]
f_stat_loc, p_value_loc = f_oneway(*groups_loc)
print("location f statistic: ", f_stat_loc)
print("location p value: ", p_value_loc)


*   Salary varies substantially by location. USA ($$181,716) and Canada ($167,391) pays the most; India pays the least ($97,690)
*   Remote work sits in the middle of the range lower than USA/Canada etc. but higher than India
*   The dataset does not have a separate "rural" category
*   Without a true rural/ urban split, we can only conclude that some countries pay significantly more than remote work while oher pay about the same or less.

In summary, Location has a highly statistically  significant effect on salary, the strongest effect so far, even larger than education.




# Question 4
Which factor matter the most, job title, experience or skill?
Note: This will be answered definitively after modelling using feature importance since it requires comparing mulitple factors instead of one at a time.

In [ ]:
df[['experience_years', 'skills_count', 'certifications', 'salary']].corr()

In [ ]:
#visualise as heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(df[['experience_years', 'skills_count', 'certifications', 'salary']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

* experience_years shows the strongest correlation with salary (r =0.44)
* skills_count and certifications show weak/ negligible correlation with salary
* job is categorical and not captured un this numeric correlation. It will be revisited properly using feature engineering after building the model

In [ ]:
#visualise as scatterplot
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='experience_years', y='salary')
plt.xlabel('Experience (Years)')
plt.ylabel('Salary')
plt.title('Experience vs Salary')
plt.show()

#Feature Engineering
Preparing the data for modelling. Splitting into train/ test first, then encoding categorical variables

In [ ]:
X = df.drop(columns=['salary'])
y = df['salary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

In [ ]:
#education_level has a rank order
education_order = {
    'High School': 0,
    'Diploma': 1,
    'Bachelor': 2,
    'Master': 3,
    'PhD': 4
}

X_train['education_level'] = X_train['education_level'].map(education_order)
X_test['education_level'] = X_test['education_level'].map(education_order)

In [ ]:
X_train.columns.tolist()

In [ ]:
#other categories have no natural order
categorical_columns = ['job_title', 'industry', 'company_size', 'remote_work', 'location']
X_train = pd.get_dummies(X_train, columns=categorical_columns, drop_first= True)
X_test = pd.get_dummies(X_test, columns=categorical_columns,  drop_first= True)
X_train.shape, X_test.shape

Split into train and test (80 / 20) before encoding to prevent data leakage
* education_level ordinally encoded since it has natura order but other categories are one-hot encoded since they have no order
* used drop_first to avoid any redundant empty columns

# Modeling

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs= 1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
      'RSME': [
        np.sqrt(mean_squared_error(y_test, lr_pred)),
        np.sqrt(mean_squared_error(y_test, rf_pred))
    ],
    'MAE': [
        mean_absolute_error(y_test, lr_pred),
        mean_absolute_error(y_test, rf_pred)
    ],
    'R2': [
        r2_score(y_test, lr_pred),
        r2_score(y_test, rf_pred)
    ]
})
results


Both models perform very well.
The gap between the two models is small.
But Random forest is selected for slightly better accuracy and because it also provides feature importance needed for categorical analysis (number 4)

In [ ]:
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=importances.head(15), x='importance', y='feature')
plt.title = 'Top 15 Most important features (Random forest)'
plt.tight_layout()
plt.show()

importances.head(15)


## Revisting Question Four
*  experience_years is the single most important individual feature. However, location dominates when its categories are considered together. Location as a single factor would outweigh experience_years
*   education_level is the third strongest single fator and job_titles matter less on their own.

**Answer:** experience_years is the strongest single predictor, but location (particularly USA and India) is more important when considered as a whole.

# Saving the Final model

In [ ]:
import pickle

with open("salary_model.pkl", "wb") as f:
    pickle.dump(rf, f)
print ("model saved successfully")


Quick sanity Check

In [ ]:

import pickle

with open("salary_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

sample = X_test.sample(5, random_state=1)
preds = loaded_model.predict(sample)
actual = y_test.loc[sample.index]

out = df.loc[sample.index, ["job_title", "experience_years", "education_level", "location"]].copy()
out["actual_salary"] = actual
out["predicted_salary"] = preds.round(0)
out["error"] = (out["actual_salary"] - out["predicted_salary"]).round(0)
out

# Limitations
*   The dataset does not include a true rural/urban split. Location cotains "Remote" as a category. so question Three's answer about big cities vs rural is only partially adressed
*   location ranked as the top salary driver in Random Forest importance, which is not totally true since real data would expect experience or job title to dominate.

*  Categorical variables (job_title, industry, location) one-hot encoded with no regularization, some overfitting risk, though the strong test-set R² suggests it's minimal in practice.. This was noted in question one

#Conclusion
The Random Forest model developed for job salary prediction demonstrated high effectiveness with an R² of 0.9617 and an RMSE of $7,292.48. The model identified Experience Years, Location (especially USA and India), and Education Level as the most significant factors influencing salary. This model can be a valuable tool for salary benchmarking, informing recruitment strategies, and guiding career development